# Outcome model: tiny-set overfit check

Run all cells in a Colab GPU runtime. This is a separate sanity check, not a replacement for the full outcome training run. It trains a fresh copy of the existing architecture on only 32 **training** examples, one per player and eight per stay/switch × win/loss group. Validation and test outcomes are never used.

The goal is training loss near zero. That would show the network and optimizer can memorize, not that recommendations generalize or are causal.

In [ ]:
from pathlib import Path

DRIVE_PROJECT_FOLDER = Path("clash2")  # Relative to MyDrive.
REPOSITORY_URL = "https://github.com/jfbami/clash2.git"
BRANCH = "main"
SEED = 17
PER_GROUP = 8  # Four action/outcome groups, 32 distinct players total.
MAX_STEPS = 2000
TARGET_LOSS = 0.02
LEARNING_RATE = 1e-3

In [ ]:
from google.colab import drive
import torch

drive.mount("/content/drive")
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Select a Colab GPU runtime before continuing.")
print(f"GPU: {torch.cuda.get_device_name(0)}")
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive") / DRIVE_PROJECT_FOLDER
DRIVE_CACHE = DRIVE_PROJECT_ROOT / "data/switch_current_deck_count_ablation/arrays"
OUTPUT_FILE = DRIVE_PROJECT_ROOT / "data/outcome_tiny_overfit/results.json"
required = [
    DRIVE_CACHE / name
    for name in (
        "metadata.json", "cards.npy", "levels.npy", "battle_features.npy",
        "summary_features.npy", "labels.npy", "next_wins.npy",
        "splits.npy", "player_indices.npy",
    )
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing Drive cache files:\n" + "\n".join(missing))
print(f"Cache: {DRIVE_CACHE}")
print(f"Result: {OUTPUT_FILE}")

In [ ]:
import subprocess

CODE_ROOT = Path("/content/clash2_tiny_check_code")
if CODE_ROOT.exists():
    subprocess.run(["git", "-C", str(CODE_ROOT), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY_URL, str(CODE_ROOT)],
        check=True,
    )
commit = subprocess.run(
    ["git", "-C", str(CODE_ROOT), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
script = CODE_ROOT / "scripts/check_outcome_tiny_overfit.py"
if not script.exists():
    raise FileNotFoundError(f"Current main does not contain {script.name}")
print(f"Code commit: {commit}")

In [ ]:
import sys

command = [
    sys.executable, "-u", str(script),
    "--cache", str(DRIVE_CACHE),
    "--output", str(OUTPUT_FILE),
    "--per-group", str(PER_GROUP),
    "--max-steps", str(MAX_STEPS),
    "--target-loss", str(TARGET_LOSS),
    "--learning-rate", str(LEARNING_RATE),
    "--seed", str(SEED),
    "--device", "cuda",
]
print("Running:", " ".join(command))
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
result.check_returncode()

In [ ]:
import json
import pandas as pd
from IPython.display import display

report = json.loads(OUTPUT_FILE.read_text(encoding="utf-8"))
display(pd.Series({
    "examples": report["examples"],
    "distinct_players": report["distinct_players"],
    "initial_loss": report["initial_loss"],
    "final_loss": report["final_loss"],
    "target_reached": report["target_reached"],
    "steps_completed": report["steps_completed"],
    "training_accuracy": report["training_accuracy"],
}, name="tiny-set check").to_frame())
display(pd.DataFrame(report["trace"]).tail(12))
if report["target_reached"]:
    print("Sanity check passed: the model memorized these training examples.")
else:
    print("The loss did not reach the target. Inspect the trace before changing model depth.")